<a href="https://colab.research.google.com/github/elvissoares/EQE595-SimMol/blob/main/listas/Lista06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EQE595 - Modelagem e Simulação Molecular

Autor: [Prof. Elvis do A. Soares](https://github.com/elvissoares)

Contato: [elvis@peq.coppe.ufrj.br](mailto:elvis@peq.coppe.ufrj.br) - [Programa de Engenharia Química, PEQ/COPPE, UFRJ, Brasil](https://www.peq.coppe.ufrj.br/)

---

## Nome do Aluno:

## DRE:


---

### Este Notebook deve ser entregue preenchido via e-mail para elvis@peq.coppe.ufrj.br até a próxima segunda-feira (16/11) com horário limite de 23:59.

## Instruções

Esta lista usa o que foi feito na **Aula Prática 06 – OpenMM e Água**. O objetivo é obter a curva $\rho(T)$ da água a 1 bar com o modelo TIP3P, **com barras de erro corretas**, e compará-la com dados experimentais.

**Antes de começar:**

- Ative a GPU no Colab: *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)*.
- Cada temperatura leva alguns minutos de GPU. Rode primeiro **uma** temperatura para medir o tempo e só depois o laço completo.
- O Colab pode desconectar no meio do laço. Por isso o esqueleto da Questão 1 salva o resultado de cada temperatura assim que ela termina e pula as temperaturas que já foram calculadas.
- Sempre que pedirmos uma **discussão**, responda em uma célula de texto, com suas palavras, em poucas frases.

**O que entregar:** este notebook executado (com as saídas) e o arquivo `densidade_TIP3P.txt`.

# Preparação do ambiente

In [ ]:
%%capture
!pip install openmm[cuda12] mdtraj

In [ ]:
!python -m openmm.testInstallation

In [ ]:
from sys import stdout
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import openmm as mm
import openmm.app as app
import openmm.unit as unit

### Funções da Aula 06

Copie para a célula abaixo as funções da Aula 06 de que você vai precisar. A função `simula_agua` depende das outras, então copie **todas** estas:

- `escolhe_plataforma` (e execute `plataforma = escolhe_plataforma()`);
- as definições de `elemento_O` e `elemento_H`;
- `geometria_agua`, `rotacao_aleatoria` e `monta_caixa`;
- `simula_agua`, `erro_por_blocos` e `analisa`.

In [ ]:
# Cole aqui as funções da Aula 06

---
# Questão 1 — Densidade da água em função da temperatura

Calcule a densidade da água TIP3P com **1000 moléculas** a **1 bar** para as temperaturas:

- 0 °C e 4 °C;
- de 10 °C a 100 °C, com passo de 10 °C.

São 12 temperaturas no total.

Use $\Delta t = 2$ fs, 20 ps de equilibração NVT e **pelo menos 300 ps** de NPT. Para cada temperatura, registre:

| Coluna | Significado |
|---|---|
| `T (C)`, `T (K)` | temperatura |
| `rho` | densidade média (kg/m³) |
| `erro_media` | erro da média, por média em blocos |
| `flutuacao` | desvio padrão da densidade instantânea |
| `kappa_T (1e-5/bar)` | compressibilidade isotérmica, pelas flutuações de volume |
| `U/N` | energia potencial média por molécula (kJ/mol) |

Salve a tabela no arquivo `densidade_TIP3P.txt`.

> 💡 Antes de rodar o laço completo, rode **uma** temperatura e confira o gráfico de $\rho(t)$. O tempo de corte da equilibração (`t_corte`) e o tamanho de bloco (`bloco_ps`) da Aula 06 continuam adequados? Justifique.

In [ ]:
T_celsius = np.concatenate([[0, 4], np.arange(10, 101, 10)])
print("Temperaturas (°C):", T_celsius)

arquivo_saida = "densidade_TIP3P.txt"
if os.path.exists(arquivo_saida):
    tabela = pd.read_csv(arquivo_saida, sep="\t")      # continua de onde parou
else:
    tabela = pd.DataFrame()

for Tc in T_celsius:
    if len(tabela) > 0 and Tc in tabela["T (C)"].values:
        print(f"{Tc} °C já calculado, pulando")
        continue
    TK = Tc + 273.15
    print(f"\n===== T = {Tc} °C =====")

    # TODO: rode simula_agua(...) nesta temperatura e analise o resultado com analisa(...)
    res = ...

    res["T (C)"], res["T (K)"] = Tc, TK
    tabela = pd.concat([tabela, pd.DataFrame([res])], ignore_index=True)
    tabela.to_csv(arquivo_saida, sep="\t", index=False)   # salva a cada temperatura

tabela

**Discussão 1.** A coluna `flutuacao` é bem maior que `erro_media`. Explique a diferença entre as duas grandezas. Qual delas diminuiria se você dobrasse o tempo de simulação? E qual diminuiria se você dobrasse o número de moléculas?

*Sua resposta aqui.*

---
# Questão 2 — Comparação com o NIST

Os dados experimentais do NIST a 1 bar estão no arquivo `water_NIST.txt`, na pasta `listas` do GitHub do curso. Nele, a densidade está em **mol/m³**.

Ref: https://webbook.nist.gov/cgi/fluid.cgi?P=0.1&TLow=273.16&THigh=372.76&TInc=1&Digits=5&ID=C7732185&Action=Load&Type=IsoBar&TUnit=K&PUnit=MPa&DUnit=mol%2Fm3&HUnit=kJ%2Fmol&WUnit=m%2Fs&VisUnit=uPa*s&STUnit=N%2Fm&RefState=DEF

Faça um gráfico da densidade molar (mol/m³) em função da temperatura com:
- os pontos da simulação **com barras de erro** (erro da média);
- a curva do NIST.

> 💡 Inspecione o arquivo (`nist.head()`, `nist.columns`) antes de usá-lo: confira o separador e os nomes das colunas.

In [ ]:
!wget -q -nc https://raw.githubusercontent.com/elvissoares/EQE595-SimMol/main/listas/water_NIST.txt
nist = pd.read_csv("water_NIST.txt", sep="\t")
nist.head()

In [ ]:
# Seu código aqui: converta rho (kg/m³) da simulação para mol/m³ e faça o gráfico

**Discussão 2.**
1. O TIP3P reproduz a **tendência** da densidade com a temperatura? E os **valores absolutos**?
2. Os desvios em relação ao NIST são maiores ou menores que as barras de erro? O que isso indica sobre a origem dos desvios: estatística ou modelo?

*Sua resposta aqui.*

---
# Questão 3 — Densidade de massa

Converta os dados do gráfico (simulação, com as barras de erro, e NIST) para densidade de massa em kg/m³ e refaça o gráfico. Num segundo painel, mostre o desvio relativo $(\rho_\text{sim}-\rho_\text{NIST})/\rho_\text{NIST}$ em %, interpolando os dados do NIST nas temperaturas simuladas (`np.interp`).

In [ ]:
# Seu código aqui

---
# Questão 4 — Expansão térmica e o máximo de densidade

**a)** O coeficiente de expansão térmica isobárico é
$$\alpha_P = \frac{1}{V}\left(\frac{\partial V}{\partial T}\right)_P = -\frac{1}{\rho}\left(\frac{\partial \rho}{\partial T}\right)_P.$$
Ajuste um polinômio de grau 2 ou 3 aos seus dados de $\rho(T)$ entre 10 °C e 100 °C (`np.polyfit`) e calcule $\alpha_P(T)$. Faça o mesmo com os dados do NIST e compare as duas curvas num gráfico.

**b)** A água real tem um máximo de densidade em ≈ 4 °C. Entre 0 °C e 4 °C, a densidade experimental varia apenas cerca de 0,1 kg/m³.
- Compare as densidades simuladas a 0 °C e 4 °C. Considerando as barras de erro, é possível dizer se o TIP3P tem (ou não) um máximo nessa faixa?
- Sabendo que o erro da média cai como $1/\sqrt{t_\text{NPT}}$, estime quanto tempo de NPT seria necessário, **por temperatura**, para resolver uma diferença de 0,1 kg/m³.

In [ ]:
# Seu código aqui

*Sua resposta aqui.*

---
# Questão 5 (bônus) — Compressibilidade isotérmica

Faça um gráfico de $\kappa_T(T)$ obtido pelas flutuações de volume (coluna `kappa_T (1e-5/bar)` da Questão 1).

Para comparar com o experimento, use as relações termodinâmicas
$$\kappa_S = \frac{1}{\rho_m\,w^2}, \qquad \kappa_T = \frac{C_P}{C_V}\,\kappa_S,$$
onde $\rho_m$ é a densidade de massa, $w$ é a velocidade do som e $C_P/C_V$ é a razão das capacidades caloríficas. Todas essas grandezas estão nos dados do NIST.
- A água real tem um **mínimo** de $\kappa_T$ em ≈ 46 °C. O TIP3P reproduz esse comportamento?
- Compare a dispersão dos seus pontos de $\kappa_T$ com a dos pontos de $\rho$. Por que flutuações convergem mais devagar que médias?

In [ ]:
# Seu código aqui

*Sua resposta aqui.*